# NV5 — Dự báo occupancy cổng và session arrivals

Toàn bộ mã nguồn NV5 nằm trong notebook này; thư mục scripts dành cho chương 4. Các cell dưới đây nạp dữ liệu NV1, tạo chuỗi theo cụm trạm, chọn tham số bằng validation, fit baseline/mô hình và xuất artefact. Đọc reports/NV5_demand_forecasting.md để xem định nghĩa mục 3.5.1–3.5.2.

Trước khi chạy cell cuối, nhóm cần điền crosswalk station_name,cluster_id đã thống nhất vào data/processed/station_cluster_map.csv, đồng thời tạo reports/nv5_splits.json theo mốc train/validation/test do Khải cung cấp. Notebook không tự quyết định cluster hay cách chia tập.

## 1. Thiết lập và thư viện

In [8]:
"""NV5: 15-minute station-cluster port occupancy and session-arrival forecasts.

Run each notebook cell in order after the team supplies the approved station-cluster
mapping and split milestones.

The station-name crosswalk must contain station_name,cluster_id. Each session is grouped by its recorded location, so relocated ports can change site cluster over time. The split JSON must contain
train_end_utc, validation_end_utc, and test_end_utc (exclusive boundaries).
"""
from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
while not (ROOT / 'src' / 'config.py').exists():
    if ROOT.parent == ROOT:
        raise RuntimeError('Open this notebook from the project workspace')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.config import (RAW_DATA_FILE, SESSIONS_CLEAN_PARQUET,
                        SESSIONS_FEATURES_PARQUET)

OUT = ROOT / "outputs" / "nv5"
STEP = pd.Timedelta(minutes=15)
SEED = 42


def _as_datetime(series):
    return pd.to_datetime(series, errors="coerce", utc=True).dt.tz_convert(None)


def _expected_port_id(df):
    # Chuẩn hóa dấu phân tách và chữ hoa/thường để gộp các cách viết cùng địa chỉ MAC.
    mac = (df["MAC Address"].astype("string").str.strip().str.upper()
           .str.replace(r"[^0-9A-F]", "", regex=True))
    port = df["Port Number"].astype("string").str.strip()
    good = mac.notna() & port.notna() & mac.ne("") & port.ne("")
    result = pd.Series(pd.NA, index=df.index, dtype="string")
    result.loc[good] = mac.loc[good] + "_Port_" + port.loc[good]
    return result


def _normalize_station_name(values):
    """Normalize whitespace and case for matching station-name crosswalk entries."""
    return (values.astype("string").str.upper().str.strip()
            .str.replace(r"\s+", " ", regex=True))


def _normalize_port_id(values):
    """Canonicalize MAC punctuation/case while preserving the physical port number."""
    text = values.astype("string").str.strip()
    parts = text.str.extract(r"^(.*)_Port_([^_]+)$")
    mac = parts[0].astype("string").str.upper().str.replace(r"[^0-9A-F]", "", regex=True)
    port = parts[1].astype("string").str.strip()
    good = mac.notna() & port.notna() & mac.ne("") & port.ne("")
    result = pd.Series(pd.NA, index=values.index, dtype="string")
    result.loc[good] = mac.loc[good] + "_Port_" + port.loc[good]
    return result


def _normalize_station_name(values):
    """Normalize whitespace and case for matching station-name crosswalk entries."""
    return (values.astype("string").str.upper().str.strip()
            .str.replace(r"\s+", " ", regex=True))


## 2. Nạp dữ liệu NV1, kiểm tra port_id và gán cluster

In [9]:
def load_sessions():
    """Prefer NV1 features, compare its row/valid counts with clean parquet."""
    parquet_rejection = None
    if SESSIONS_FEATURES_PARQUET.exists():
        try:
            features = pd.read_parquet(SESSIONS_FEATURES_PARQUET)
            clean = (pd.read_parquet(SESSIONS_CLEAN_PARQUET)
                     if SESSIONS_CLEAN_PARQUET.exists() else None)
            if clean is not None:
                k = ["MAC Address", "Port Number", "Start Date", "End Date"]
                audit = {
                    "clean_rows": len(clean), "features_rows": len(features),
                    "clean_valid_sessions": int(clean["is_valid_session"].sum()),
                    "features_valid_sessions": int(features["is_valid_session"].sum()),
                    "same_session_keys": bool(clean[k].sort_values(k).reset_index(drop=True).equals(
                        features[k].sort_values(k).reset_index(drop=True))),
                }
                if (audit["clean_rows"] != audit["features_rows"] or
                    audit["clean_valid_sessions"] != audit["features_valid_sessions"] or
                    not audit["same_session_keys"]):
                    raise ValueError(f"NV1 clean/features mismatch: {audit}")
            else:
                audit = {"clean_parquet_available": False}
            print("Source: NV1 sessions_features.parquet; audit:", audit)
            return features, "nv1_parquet", audit
        except (ImportError, ValueError, OSError) as exc:
            parquet_rejection = str(exc)
            print("NV1 parquet unavailable/inconsistent:", str(exc)[:300])

    from src.feature_engineering import build_clean_pipeline
    print("Fallback: rebuild NV1 cleaning flags from raw CSV")
    raw = pd.read_csv(RAW_DATA_FILE, low_memory=False)
    data, initial, duplicates = build_clean_pipeline(raw)
    audit = {"raw_rows": initial, "exact_duplicates_removed": duplicates,
             "clean_rows": len(data),
             "valid_sessions": int(data["is_valid_session"].sum()),
             "parquet_rejection_reason": parquet_rejection}
    return data, "nv1_raw_recomputed", audit


def attach_clusters(df, mapping_path):
    """Assign a location cluster per session from its station-name crosswalk."""
    if "MAC Address" not in df or "Port Number" not in df:
        raise ValueError("NV1 source must include MAC Address and Port Number")
    expected = _expected_port_id(df)
    if "port_id" in df:
        raw_actual = df["port_id"].astype("string").str.strip()
        actual = _normalize_port_id(df["port_id"])
        populated_actual = raw_actual.notna() & raw_actual.ne("")
        mismatch = expected.notna() & populated_actual & (actual.isna() | expected.ne(actual))
        if mismatch.any():
            raise ValueError(f"port_id disagrees with MAC Address + Port Number on {int(mismatch.sum())} rows")
        missing_components_with_id = expected.isna() & actual.notna()
        if missing_components_with_id.any():
            raise ValueError("port_id is populated where MAC Address/Port Number is missing")
    df = df.copy()
    df["port_id"] = expected
    co_id = df.port_id.notna()
    mac_chuan = (df.loc[co_id, "MAC Address"].astype("string").str.strip().str.upper()
                 .str.replace(r"[^0-9A-F]", "", regex=True))
    pairs = df.loc[co_id, ["port_id", "Port Number"]].assign(mac=mac_chuan).drop_duplicates()
    if pairs.port_id.duplicated().any():
        raise ValueError("A port_id maps to multiple physical MAC Address + Port Number pairs")
    pair_to_id = pairs.groupby(["mac", "Port Number"], dropna=False).port_id.nunique()
    if pair_to_id.gt(1).any():
        raise ValueError("A physical MAC Address + Port Number pair maps to multiple port_id values")

    mapping_path = Path(mapping_path)
    station_key = _normalize_station_name(df["Station Name"])
    if not mapping_path.exists():
        template = pd.DataFrame({"station_name": sorted(station_key.dropna().unique()), "cluster_id": ""})
        mapping_path.parent.mkdir(parents=True, exist_ok=True)
        template.to_csv(mapping_path, index=False)
        raise FileNotFoundError(
            f"Created blank station-name crosswalk at {mapping_path}; have the group assign cluster_id values"
        )
    mapping = pd.read_csv(mapping_path, dtype="string")
    required = {"station_name", "cluster_id"}
    if not required.issubset(mapping.columns):
        raise ValueError("Station-cluster mapping requires columns station_name,cluster_id")
    mapping = mapping[["station_name", "cluster_id"]].copy()
    mapping["station_name"] = _normalize_station_name(mapping["station_name"])
    mapping["cluster_id"] = mapping.cluster_id.astype("string").str.strip()
    if mapping.isna().any().any() or mapping.eq("").any().any():
        raise ValueError("Station-cluster mapping contains blank station_name/cluster_id values")
    if mapping.station_name.duplicated().any():
        raise ValueError("Each normalized Station Name must map to exactly one cluster_id")
    known_names = set(station_key.dropna())
    unknown = set(mapping.station_name) - known_names
    if unknown:
        raise ValueError(f"Cluster mapping contains {len(unknown)} Station Names absent from NV1 data")
    unmapped = known_names - set(mapping.station_name)
    if unmapped:
        raise ValueError(f"Cluster mapping is missing {len(unmapped)} observed Station Names")
    df["cluster_id"] = station_key.map(mapping.set_index("station_name").cluster_id)
    df.attrs["cluster_mapping"] = {"mapped_station_names": int(mapping.station_name.nunique()),
                                     "cluster_ids": sorted(mapping.cluster_id.unique().tolist()),
                                     "port_ids_with_multiple_location_clusters": int(
                                         df.dropna(subset=["port_id"]).groupby("port_id").cluster_id.nunique().gt(1).sum())}
    return df


def make_regular_series(df):
    """Create cluster-local counts on a regular UTC grid using [start,end)."""
    required = ["start_datetime_utc", "end_datetime_utc", "is_valid_session",
                "port_id", "cluster_id"]
    absent = [c for c in required if c not in df]
    if absent:
        raise ValueError(f"Missing required columns: {absent}")
    valid_source = df.loc[df.is_valid_session.fillna(False)].copy()
    missing_port_count = int(valid_source.port_id.isna().sum())
    missing_cluster_count = int(valid_source.cluster_id.isna().sum())
    data = valid_source.copy()
    data["start_datetime_utc"] = _as_datetime(data.start_datetime_utc)
    data["end_datetime_utc"] = _as_datetime(data.end_datetime_utc)
    nonpositive_intervals = int((data.end_datetime_utc <= data.start_datetime_utc).fillna(False).sum())
    data = data.dropna(subset=["start_datetime_utc", "end_datetime_utc", "cluster_id"])
    data = data.loc[data.end_datetime_utc > data.start_datetime_utc]
    if data.empty:
        raise ValueError("No valid, mapped sessions remain")
    left = data.start_datetime_utc.min().floor("15min")
    right = data.end_datetime_utc.max().ceil("15min")
    index = pd.date_range(left, right - STEP, freq="15min")
    frames = []
    for cluster, group in data.groupby("cluster_id", sort=True):
        # Invalid/missing port IDs do not count toward physical occupancy.
        known = group.loc[group.port_id.notna() & group.port_id.astype("string").str.strip().ne("")]
        occupancy = np.zeros(len(index), dtype=np.int64)
        # Each port contributes at most 1 in a bin, even when sessions overlap.
        for _, port_sessions in known.groupby("port_id", sort=False):
            diffs = np.zeros(len(index) + 1, dtype=np.int64)
            starts = ((port_sessions.start_datetime_utc - left) // STEP).to_numpy(dtype=int)
            ends = ((port_sessions.end_datetime_utc - left - pd.Timedelta(1, "ns")) // STEP + 1).to_numpy(dtype=int)
            starts = np.clip(starts, 0, len(index))
            ends = np.clip(ends, 0, len(index))
            np.add.at(diffs, starts, 1)
            np.add.at(diffs, ends, -1)
            occupancy += np.cumsum(diffs[:-1]) > 0

        arrivals = np.zeros(len(index), dtype=np.int64)
        starts = ((group.start_datetime_utc - left) // STEP).to_numpy(dtype=int)
        in_range = (starts >= 0) & (starts < len(index))
        np.add.at(arrivals, starts[in_range], 1)
        # Keep one shared calendar grid for every cluster, including leading/trailing zeros.
        frames.append(pd.DataFrame({"cluster_id": cluster, "timestamp_utc": index,
                                    "occupancy_ports": occupancy,
                                    "session_arrivals": arrivals,
                                    "observed_physical_ports": int(known.port_id.nunique())}))
    result = pd.concat(frames, ignore_index=True)
    info = {"valid_mapped_sessions": int(len(data)),
            "valid_sessions_excluded_nonpositive_interval": nonpositive_intervals,
            "valid_sessions_missing_port_id": missing_port_count,
            "valid_sessions_without_cluster_id": missing_cluster_count,
            "observed_physical_ports": int(data.port_id.nunique()),
            "first_utc": str(left), "last_utc_exclusive": str(right),
            "time_bins_per_cluster_min": int(result.groupby("cluster_id").size().min()),
            "time_bins_per_cluster_max": int(result.groupby("cluster_id").size().max()),
            "clusters": sorted(data.cluster_id.astype(str).unique().tolist()),
            "max_occupancy": int(result.occupancy_ports.max()),
            "occupancy_exceeds_observed_ports": bool((result.occupancy_ports > result.observed_physical_ports).any())}
    if info["occupancy_exceeds_observed_ports"]:
        raise AssertionError("Cluster occupancy exceeds its observed physical port count")
    return result, info


## 3. Tạo chuỗi occupancy và arrivals theo cụm

In [10]:
def add_features(series, target):
    frames = []
    for cluster, group in series.groupby("cluster_id", sort=False):
        g = group.sort_values("timestamp_utc").set_index("timestamp_utc")
        s = g[[target]].copy()
        t = s.index
        slot = t.hour * 4 + t.minute // 15
        dow = t.day_of_week
        s["slot"] = slot.astype(int)
        s["dow"] = dow.astype(int)
        s["month"] = t.month.astype(int)
        s["year"] = t.year.astype(int)
        s["weekend"] = (dow >= 5).astype(int)
        s["post_policy_2017"] = (t >= pd.Timestamp("2017-08-01")).astype(int)
        s["hour_sin"] = np.sin(2*np.pi*slot/96)
        s["hour_cos"] = np.cos(2*np.pi*slot/96)
        s["week_sin"] = np.sin(2*np.pi*dow/7)
        s["week_cos"] = np.cos(2*np.pi*dow/7)
        for lag in (1, 2, 4, 96, 672):
            s[f"lag_{lag}"] = s[target].shift(lag)
        for window in (4, 12, 96, 672):
            s[f"rolling_mean_{window}"] = s[target].shift(1).rolling(window, min_periods=window).mean()
        s = s.dropna().copy()
        s["cluster_id"] = cluster
        frames.append(s.reset_index(names="timestamp_utc"))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


## 4. Đặc trưng lịch, lag/rolling (chỉ dùng quá khứ) và hàm đánh giá

In [11]:
def metrics(y, p):
    return {"MAE": float(mean_absolute_error(y, p)),
            "RMSE": float(np.sqrt(mean_squared_error(y, p))),
            "R2": float(r2_score(y, p))}


def _parse_splits(splits):
    train_end, validation_end, test_end = [
        pd.to_datetime(splits[key], utc=True).tz_convert(None)
        for key in ("train_end_utc", "validation_end_utc", "test_end_utc")
    ]
    if not train_end < validation_end < test_end:
        raise ValueError("Require train_end_utc < validation_end_utc < test_end_utc")
    stress_start = (pd.to_datetime(splits["stress_start_utc"], utc=True).tz_convert(None)
                    if splits.get("stress_start_utc") else None)
    if stress_start is not None and stress_start < test_end:
        raise ValueError("stress_start_utc must be on or after test_end_utc")
    return train_end, validation_end, test_end, stress_start


def ridge_predictions(train, validation, test, features, target, alpha=10.0):
    """Fit the scaler and Ridge only on train; transform later splits unchanged."""
    scaler = StandardScaler().fit(train[features])
    model = Ridge(alpha=alpha).fit(scaler.transform(train[features]), train[target])
    return (model.predict(scaler.transform(validation[features])),
            model.predict(scaler.transform(test[features])), scaler)


def clip_to_port_limit(predictions, port_limit=None):
    """Keep counts nonnegative; cap occupancy only when a port-count bound applies."""
    values = np.maximum(0.0, np.asarray(predictions, dtype=float))
    return np.minimum(values, float(port_limit)) if port_limit is not None else values


def run_target(series, target, splits):
    """Run per-cluster baselines/models with externally supplied split boundaries."""
    from xgboost import XGBRegressor
    train_end, validation_end, test_end, stress_start = _parse_splits(splits)
    stress_end = (pd.to_datetime(splits.get("stress_end_utc", "2021-01-01T00:00:00Z"), utc=True).tz_convert(None)
                  if stress_start is not None else None)
    if stress_start is not None and stress_end <= stress_start:
        raise ValueError("stress_end_utc must be after stress_start_utc")
    records, prediction_rows = [], []
    # Build features one cluster at a time to keep memory bounded on the full 15-minute grid.
    for cluster, raw in series.groupby("cluster_id", sort=True):
        d = add_features(raw, target).sort_values("timestamp_utc")
        cols = [c for c in d.columns if c not in {target, "cluster_id", "timestamp_utc"}]
        train = d.loc[d.timestamp_utc < train_end]
        val = d.loc[(d.timestamp_utc >= train_end) & (d.timestamp_utc < validation_end)]
        test = d.loc[(d.timestamp_utc >= validation_end) & (d.timestamp_utc < test_end)]
        stress = (d.loc[(d.timestamp_utc >= stress_start) & (d.timestamp_utc < stress_end)] if stress_start is not None
                  else d.iloc[0:0].copy())
        if stress_start is not None and stress.empty:
            raise ValueError(f"Cluster {cluster!r} has no observations in the stress-test period")
        if train.empty or val.empty or test.empty:
            raise ValueError(f"Cluster {cluster!r} has an empty train/validation/test split")
        hist = train.groupby(["dow", "slot"])[target].mean()
        default = float(train[target].mean())
        # No as-of capacity history is available; a full-sample port count would leak future installations.
        prediction_upper_bound = None
        split_parts = [("validation", val), ("test", test)]
        if stress_start is not None:
            split_parts.append(("stress_2020", stress))
        prediction_sets = {
            "Historical Mean": {part: np.array([hist.get((int(row.dow), int(row.slot)), default)
                                             for row in part_d.itertuples()])
                                for part, part_d in split_parts},
            "Persistence Step t-1": {part: part_d.lag_1.to_numpy() for part, part_d in split_parts},
            "Persistence Day t-96": {part: part_d.lag_96.to_numpy() for part, part_d in split_parts},
            "Persistence Week t-672": {part: part_d.lag_672.to_numpy() for part, part_d in split_parts},
        }
        selected_parameters = {}
        ridge_candidates = []
        for alpha in (0.1, 1.0, 10.0, 100.0):
            pred_val, pred_test, _ = ridge_predictions(train, val, test, cols, target, alpha=alpha)
            candidate_rmse = float(np.sqrt(mean_squared_error(
                val[target], clip_to_port_limit(pred_val, prediction_upper_bound))))
            ridge_candidates.append((candidate_rmse, alpha, pred_val, pred_test))
        _, best_alpha, pred_val, pred_test = min(ridge_candidates, key=lambda item: item[0])
        selected_parameters["Ridge"] = {"alpha": best_alpha}
        prediction_sets["Ridge"] = {"validation": clip_to_port_limit(pred_val, prediction_upper_bound),
                                     "test": clip_to_port_limit(pred_test, prediction_upper_bound)}
        if stress_start is not None:
            scaler = StandardScaler().fit(train[cols])
            selected_ridge = Ridge(alpha=best_alpha).fit(scaler.transform(train[cols]), train[target])
            prediction_sets["Ridge"]["stress_2020"] = clip_to_port_limit(
                selected_ridge.predict(scaler.transform(stress[cols])), prediction_upper_bound)

        # Fit one bounded forest per cluster; release it before XGBoost tuning.
        rf_params = {"n_estimators": 60, "max_depth": 12, "min_samples_leaf": 4}
        rf = RandomForestRegressor(**rf_params, n_jobs=2, random_state=SEED)
        rf.fit(train[cols], train[target])
        prediction_sets["Random Forest"] = {
            "validation": clip_to_port_limit(rf.predict(val[cols]), prediction_upper_bound),
            "test": clip_to_port_limit(rf.predict(test[cols]), prediction_upper_bound),
        }
        if stress_start is not None:
            prediction_sets["Random Forest"]["stress_2020"] = clip_to_port_limit(
                rf.predict(stress[cols]), prediction_upper_bound)
        selected_parameters["Random Forest"] = rf_params
        del rf

        # Validation selects among a small prespecified grid; keep only the current best model.
        best_xgb, best_params, best_pred_val, best_rmse = None, None, None, float("inf")
        for params in (
            {"n_estimators": 100, "max_depth": 3, "learning_rate": 0.05},
            {"n_estimators": 160, "max_depth": 4, "learning_rate": 0.05},
        ):
            model = XGBRegressor(**params, subsample=.85, colsample_bytree=.9,
                tree_method="hist", objective="reg:squarederror", n_jobs=2, random_state=SEED)
            model.fit(train[cols], train[target])
            pred_val_candidate = clip_to_port_limit(model.predict(val[cols]), prediction_upper_bound)
            candidate_rmse = float(np.sqrt(mean_squared_error(val[target], pred_val_candidate)))
            if candidate_rmse < best_rmse:
                best_xgb, best_params, best_pred_val, best_rmse = model, params, pred_val_candidate, candidate_rmse
            del model
        prediction_sets["XGBoost"] = {"validation": best_pred_val,
            "test": clip_to_port_limit(best_xgb.predict(test[cols]), prediction_upper_bound)}
        if stress_start is not None:
            prediction_sets["XGBoost"]["stress_2020"] = clip_to_port_limit(
                best_xgb.predict(stress[cols]), prediction_upper_bound)
        selected_parameters["XGBoost"] = best_params
        for name, parts in prediction_sets.items():
            for split_name, part in split_parts:
                pred = clip_to_port_limit(parts[split_name], prediction_upper_bound)
                y = part[target].to_numpy(dtype=float)
                records.append({"cluster_id": cluster, "target": target, "model": name,
                                "split": split_name, "train_rows": len(train),
                                "validation_rows": len(val), "test_rows": len(test),
                                "train_end_utc": str(train_end), "validation_end_utc": str(validation_end),
                                "test_end_utc": str(test_end),
                                "selected_parameters": json.dumps(selected_parameters.get(name, {}), sort_keys=True),
                                **metrics(y, pred)})
                prediction_rows.extend({"cluster_id": cluster,
                    "timestamp_utc": ts.isoformat(), "target": target, "model": name,
                    "split": split_name, "y_true": float(actual), "y_pred": float(estimate)}
                    for ts, actual, estimate in zip(part.timestamp_utc, y, pred))
        del d, train, val, test, stress, best_xgb
    return pd.DataFrame(records), pd.DataFrame(prediction_rows)


## 5. Baseline, Ridge, Random Forest và XGBoost


In [12]:
def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--cluster-map", required=True,
                        help="Group-approved CSV with station_name,cluster_id")
    parser.add_argument("--splits-json", required=True,
                        help="JSON with train_end_utc, validation_end_utc, test_end_utc")
    args = parser.parse_args(argv)
    OUT.mkdir(parents=True, exist_ok=True)
    df, provenance, source_audit = load_sessions()
    df = attach_clusters(df, args.cluster_map)
    series, series_audit = make_regular_series(df)
    with open(args.splits_json, encoding="utf-8") as f:
        train_end, validation_end, test_end, _ = _parse_splits(json.load(f))
    if test_end > series.timestamp_utc.max() + STEP:
        raise ValueError(f"test_end_utc {test_end} is after the last 15-minute bin")
    if train_end < series.timestamp_utc.min():
        raise ValueError(f"train_end_utc {train_end} is before the first 15-minute bin")
    series_audit.update({"source": provenance, **source_audit,
                         "cluster_mapping": df.attrs.get("cluster_mapping", {})})
    pd.Series(series_audit).to_json(OUT / "data_audit.json", indent=2, force_ascii=False)
    series[["cluster_id", "timestamp_utc", "occupancy_ports", "session_arrivals"]].to_csv(
        OUT / "timeseries_15min.csv", index=False)
    with open(args.splits_json, encoding="utf-8") as f:
        splits = json.load(f)
    score_parts, prediction_parts = [], []
    for target in ("occupancy_ports", "session_arrivals"):
        scores, predictions = run_target(series, target, splits)
        score_parts.append(scores)
        prediction_parts.append(predictions)
    comparison = pd.concat(score_parts, ignore_index=True)
    predictions = pd.concat(prediction_parts, ignore_index=True)
    comparison.to_csv(OUT / "nv5_model_comparison.csv", index=False)
    predictions.to_csv(OUT / "baseline_predictions.csv", index=False)
    print("NV5 metrics:\n", comparison.to_string(index=False))
    print("Artifacts:", OUT)


## 6. Kiểm tra dữ liệu đầu vào (chạy được ngay)

Cell này hiện kết quả từ NV1 và lưu audit mà không cần mapping cụm hay mốc chia. Đây là kiểm tra dữ liệu, chưa phải metric dự báo theo cụm.


In [13]:
sessions, data_source, source_audit = load_sessions()
valid_sessions = sessions.loc[sessions.is_valid_session.fillna(False)].copy()
valid_start = _as_datetime(valid_sessions["start_datetime_utc"])
valid_end = _as_datetime(valid_sessions["end_datetime_utc"])
expected_ids = _expected_port_id(sessions)
stored_ids = _normalize_port_id(sessions["port_id"])
id_comparable = expected_ids.notna() & stored_ids.notna()
id_mismatches = int((expected_ids.loc[id_comparable] != stored_ids.loc[id_comparable]).sum())
port_pairs = sessions.loc[sessions.port_id.notna(), ["port_id", "MAC Address", "Port Number"]].drop_duplicates()
port_pair_conflicts = int(port_pairs.groupby("port_id")[["MAC Address", "Port Number"]].nunique().gt(1).any(axis=1).sum())
cluster_map_path = ROOT / "data" / "processed" / "station_cluster_map.csv"
splits_path = ROOT / "reports" / "nv5_splits.json"
cluster_map_ready = False
if cluster_map_path.exists():
    map_preview = pd.read_csv(cluster_map_path, dtype="string")
    map_has_values = ({"station_name", "cluster_id"}.issubset(map_preview.columns)
                      and not map_preview[["station_name", "cluster_id"]].isna().any().any()
                      and not map_preview[["station_name", "cluster_id"]].apply(lambda c: c.str.strip().eq("")).any().any())
    if map_has_values:
        try:
            attach_clusters(sessions, cluster_map_path)
            cluster_map_ready = True
        except (ValueError, KeyError) as error:
            cluster_map_validation_error = str(error)
    else:
        cluster_map_validation_error = "Mapping thiếu cột station_name,cluster_id hoặc còn ô rỗng"
audit = {
    "data_source": data_source,
    **source_audit,
    "valid_sessions": int(len(valid_sessions)),
    "valid_zero_duration_sessions": int((valid_end == valid_start).sum()),
    "valid_negative_duration_sessions": int((valid_end < valid_start).sum()),
    "valid_sessions_missing_port_id": int(valid_sessions.port_id.isna().sum()),
    "source_port_id_spellings": int(sessions.port_id.nunique()),
    "observed_port_ids": int(expected_ids.nunique()),
    "port_id_mismatches_vs_mac_port": id_mismatches,
    "port_ids_mapping_to_multiple_physical_pairs": port_pair_conflicts,
    "cluster_map_ready": bool(cluster_map_ready),
    "cluster_map_validation_error": locals().get("cluster_map_validation_error"),
    "split_milestones_file_present": bool(splits_path.exists()),
}
OUT.mkdir(parents=True, exist_ok=True)
pd.Series(audit).to_json(OUT / "nv5_input_audit.json", indent=2, force_ascii=False)
from IPython.display import display
display(pd.DataFrame({"check": list(audit), "result": list(audit.values())}))
print("Input audit saved to:", OUT / "nv5_input_audit.json")


Source: NV1 sessions_features.parquet; audit: {'clean_rows': 259411, 'features_rows': 259411, 'clean_valid_sessions': 259356, 'features_valid_sessions': 259356, 'same_session_keys': True}


,check,result
0,data_source,nv1_parquet
1,clean_rows,259411
2,features_rows,259411
3,clean_valid_sessions,259356
4,features_valid_sessions,259356
5,same_session_keys,True
6,valid_sessions,259356
7,valid_zero_duration_sessions,12
8,valid_negative_duration_sessions,0
9,valid_sessions_missing_port_id,0


Input audit saved to: /home/kedora/SEM5/ba/project/ev-charging-analytics---IS403/outputs/nv5/nv5_input_audit.json


## 7. Chạy thí nghiệm và lưu kết quả

Chạy cell sau cùng sau khi hai đầu vào của nhóm đã sẵn sàng. Kết quả được ghi vào outputs/nv5/.


In [14]:
cluster_map = ROOT / 'data' / 'processed' / 'station_cluster_map.csv'
splits_json = ROOT / 'reports' / 'nv5_splits.json'
missing_inputs = []
if not cluster_map.exists():
    missing_inputs.append(f'Chưa có {cluster_map}. Sao chép reports/station_cluster_map_template.csv tới vị trí này và điền cluster_id theo 9 nhóm vị trí đã thống nhất.')
if not splits_json.exists():
    missing_inputs.append(f'Chưa có {splits_json}. Tạo JSON với train_end_utc, validation_end_utc, test_end_utc theo mốc Khải cung cấp; thêm stress_start_utc nếu đánh giá stress test 2020.')
if cluster_map.exists():
    mapping_check = pd.read_csv(cluster_map, dtype='string')
    required_map_columns = {'station_name', 'cluster_id'}
    if not required_map_columns.issubset(mapping_check.columns):
        missing_inputs.append('Cluster map phải có hai cột station_name,cluster_id.')
    elif mapping_check[list(required_map_columns)].isna().any().any() or mapping_check[list(required_map_columns)].apply(lambda col: col.str.strip().eq('')).any().any():
        missing_inputs.append('Cluster map còn station_name/cluster_id trống; hãy điền đầy đủ theo quyết định của nhóm.')
if splits_json.exists():
    try:
        with open(splits_json, encoding='utf-8') as split_file:
            _parse_splits(json.load(split_file))
    except (KeyError, ValueError, TypeError) as error:
        missing_inputs.append(f'Mốc chia trong {splits_json} chưa hợp lệ: {error}')
if missing_inputs:
    print('Chưa chạy NV5 vì còn thiếu đầu vào bắt buộc:')
    print('\n'.join(f'- {message}' for message in missing_inputs))
else:
    main(['--cluster-map', str(cluster_map), '--splits-json', str(splits_json)])


Source: NV1 sessions_features.parquet; audit: {'clean_rows': 259411, 'features_rows': 259411, 'clean_valid_sessions': 259356, 'features_valid_sessions': 259356, 'same_session_keys': True}
NV5 metrics:
    cluster_id           target                  model       split  train_rows  validation_rows  test_rows       train_end_utc  validation_end_utc        test_end_utc                                          selected_parameters      MAE     RMSE        R2
       BRYANT  occupancy_ports        Historical Mean  validation      259667            17376      17664 2019-01-01 00:00:00 2019-07-01 00:00:00 2020-01-01 00:00:00                                                           {} 1.460789 2.070171  0.365389
       BRYANT  occupancy_ports        Historical Mean        test      259667            17376      17664 2019-01-01 00:00:00 2019-07-01 00:00:00 2020-01-01 00:00:00                                                           {} 1.416162 2.003309  0.383058
       BRYANT  occupancy_ports   